# 대규모 데이터 처리 (Google Colab 버전)

이 노트북은 Google Colab 환경에서 대규모 데이터 처리 예제를 실행할 수 있도록 수정되었습니다.
기존 예제에서 사용된 라이브러리를 설치하고, 로컬 파일 시스템 및 외부 서비스(HDFS, Kafka)에 대한 의존성을 제거하거나 주석 처리했습니다.

## 1. 라이브러리 설치

예제 실행에 필요한 라이브러리들을 설치합니다.

In [ ]:
!pip -q install "dask[complete]" pyspark pyarrow fastparquet
!apt-get -qq install graphviz

## 2. Dask로 병렬 데이터 처리
Dask를 사용하여 대용량 데이터를 효율적으로 처리합니다.

In [ ]:
import time
import numpy as np
import pandas as pd
import dask
import dask.dataframe as dd
import dask.array as da
from dask.distributed import Client

print("=== Dask 기초 ===")

client = Client()
print(f"Dask 클라이언트: {client}")

def create_large_dataset(size=1_000_000):
    np.random.seed(42)
    data = {
        'id': range(size),
        'value': np.random.randn(size),
        'category': np.random.choice(['A', 'B', 'C', 'D'], size),
        'timestamp': pd.date_range('2020-01-01', periods=size, freq='1min')
    }
    return pd.DataFrame(data)

print("Pandas DataFrame 생성...")
start_time = time.time()
df_pandas = create_large_dataset()
pandas_time = time.time() - start_time
print(f"Pandas 생성 시간: {pandas_time:.2f}초")

print("Dask DataFrame 생성...")
start_time = time.time()
df_dask = dd.from_pandas(df_pandas, npartitions=4)
dask_time = time.time() - start_time
print(f"Dask 생성 시간: {dask_time:.2f}초")

print("\n=== 기본 연산 비교 ===")
start_time = time.time()
pandas_mean = df_pandas['value'].mean()
pandas_compute_time = time.time() - start_time
print(f"Pandas 평균 계산: {pandas_mean:.4f} ({pandas_compute_time:.4f}초)")

start_time = time.time()
dask_mean = df_dask['value'].mean().compute()
dask_compute_time = time.time() - start_time
print(f"Dask 평균 계산: {dask_mean:.4f} ({dask_compute_time:.4f}초)")

print("\n=== 복잡한 연산 ===")
def complex_operation(df):
    result = df.groupby('category').agg({
        'value': ['mean', 'std', 'count'],
        'id': 'nunique'
    })
    return result

start_time = time.time()
pandas_complex = complex_operation(df_pandas)
pandas_complex_time = time.time() - start_time
print(f"Pandas 복잡 연산: {pandas_complex_time:.2f}초")

start_time = time.time()
dask_complex = complex_operation(df_dask).compute()
dask_complex_time = time.time() - start_time
print(f"Dask 복잡 연산: {dask_complex_time:.2f}초")

print("\n연산 결과:")
print(dask_complex)

### Dask Array 사용

In [ ]:
print("\n=== Dask Array ===")

# Colab 환경에서 너무 큰 행렬 곱은 RAM을 터뜨릴 수 있어서 크기를 조절합니다.
# 필요하면 n, chunks를 키워서 실험하세요.
n = 3000
chunks = 500

x = da.random.random((n, n), chunks=(chunks, chunks))
y = da.random.random((n, n), chunks=(chunks, chunks))

print(f"Dask Array 형태: {x.shape}")
print(f"Chunk 크기: {x.chunksize}")

print("\n행렬 곱셈 그래프 생성...")
start_time = time.time()
z = da.dot(x, y)
graph_time = time.time() - start_time
print(f"계산 그래프 생성 시간: {graph_time:.3f}초")

print("실제 계산(compute)... (시간이 좀 걸릴 수 있음)")
start_time = time.time()
z00 = z[0, 0].compute()
compute_time = time.time() - start_time
print(f"z[0,0] = {z00:.6f}, compute 시간: {compute_time:.2f}초")

## 3. Parquet 포맷으로 저장/읽기
대용량 데이터는 CSV보다 Parquet이 훨씬 유리합니다 (압축/스키마/컬럼 단위 IO).

In [ ]:
import os
from pathlib import Path

base_dir = Path('/content/large_scale_data')
base_dir.mkdir(parents=True, exist_ok=True)
parquet_dir = base_dir / 'parquet_out'

# Dask DataFrame을 Parquet로 저장
if parquet_dir.exists():
    # Colab에서는 폴더가 남아있을 수 있어서 간단히 다른 이름 사용
    parquet_dir = base_dir / f'parquet_out_{int(time.time())}'

print('Parquet 저장 경로:', parquet_dir)
start_time = time.time()
df_dask.to_parquet(parquet_dir.as_posix(), engine='pyarrow', write_index=False)
save_time = time.time() - start_time
print(f'Parquet 저장 시간: {save_time:.2f}초')

# 다시 읽기
start_time = time.time()
df_parquet = dd.read_parquet(parquet_dir.as_posix(), engine='pyarrow')
read_time = time.time() - start_time
print(f'Parquet 로드(메타) 시간: {read_time:.2f}초')

print('컬럼:', list(df_parquet.columns))
print('head:')
print(df_parquet.head())

## 4. PySpark로 대규모 데이터 처리 (Local mode)
Colab에서는 Spark를 로컬 모드로 실행할 수 있습니다.
데이터가 더 커질수록 Spark의 분산 처리 모델(클러스터)이 강력해지지만, 여기서는 API 감각을 익히는 데 집중합니다.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .appName('LargeScaleData-Colab')
         .master('local[*]')
         .getOrCreate())

spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)

In [ ]:
# pandas -> spark dataframe
spark_df = spark.createDataFrame(df_pandas)
print('Spark DF rows:', spark_df.count())
spark_df.printSchema()
spark_df.show(5, truncate=False)

In [ ]:
# groupby aggregation
agg_df = (spark_df
          .groupBy('category')
          .agg(
              F.mean('value').alias('mean_value'),
              F.stddev('value').alias('std_value'),
              F.count('*').alias('cnt'),
              F.countDistinct('id').alias('unique_ids')
          )
          .orderBy('category'))

agg_df.show()

## 5. Spark로 Parquet 읽기
Dask가 저장한 Parquet를 Spark로도 읽어봅니다.

In [ ]:
spark_parquet_df = spark.read.parquet(parquet_dir.as_posix())
print('Spark parquet rows:', spark_parquet_df.count())
spark_parquet_df.show(5)

## 6. 마무리 & 팁
- 데이터가 커질수록 **메모리 사용량**과 **I/O 포맷**이 성능을 좌우합니다.
- Pandas는 단일 머신 메모리에 의존, Dask는 병렬화/분할, Spark는 분산 처리 모델(클러스터)을 전제로 합니다.
- Colab에서는 런타임 RAM이 제한되어 있으니, 행렬 곱/대규모 조인 같은 연산은 크기부터 조절하세요.

In [ ]:
# 리소스 정리
client.close()
spark.stop()
print('Done')